In [ ]:
# !pip cache purge

In [ ]:
!pip install numpy kokoro-onnx ipympl matplotlib 'torch>=2.11.0' 'torchaudio>=2.11.0' torchcodec misaki-fork[en]

In [ ]:
!mkdir -p models && cd models && wget -nc \
    https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin \
    https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx
!rm -rf target_voices/ && mkdir -p target_voices && cd target_voices && wget -nc \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/1/1c/Pud_spawn_03_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/2/29/Pud_spawn_09_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/8/87/Pud_battlebegins_01_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/5/59/Pud_move_06_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/9/98/Pud_ability_hook_03_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/9/94/Pud_ability_rot_10_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/3/39/Pud_ability_rot_13_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/0/09/Pud_ability_devour_15_ru.mp3 \
    https://static.wikia.nocookie.net/dota2_ru_gamepedia/images/2/2e/Pud_rival_07_ru.mp3


In [ ]:
import os
from IPython.display import display, Audio, HTML
import torch
import numpy as np

from kokoro_onnx import Kokoro
from kokoro_onnx.config import SAMPLE_RATE as KOKORO_SR
from misaki import en, espeak

from torch import hub
import torchaudio
import librosa as lr

# Вариант 6

### Задание

1. Используя CosyVoice3 (или любой другой по согласованию с преподавателем):

    - [x] озвучить текст

    - [x] oценить полученные результаты
      
1. Используя проект kNN-VC:

    - [x] сконвертировать свой голос в любой другой
  
    - выполнить дообучение вокодера (HiFi-GAN):

        - [ ] найти тестовые данные (0.5-2 часа)
  
        - [ ] сетап
  
        - [ ] сам процесс обучения (4-8 часов)
     
    - [ ] оценить, насколько хорошо метод работает в условиях ограниченных ресурсов

In [ ]:
text = '''
Speech synthesis is the artificial production of human speech. A computer system used for this purpose is called a speech synthesizer, and can be implemented in software or hardware products. A text-to-speech (TTS) system converts normal language text into speech; other systems render symbolic linguistic representations like phonetic transcriptions into speech. The reverse process is speech recognition.
'''

In [ ]:
KNN_VC_SR = 16000
def load_for_knn_vc(path):
    voice, sr = torchaudio.load(path)
    if sr != KNN_VC_SR: voice = torchaudio.functional.resample(voice, orig_freq=sr, new_freq=KNN_VC_SR)
    if voice.shape[0] > 1: voice = torch.mean(voice, dim=0, keepdim=True)
    return voice

orig_voices = list(map(load_for_knn_vc, ("./voice_by_alexei_braichuk.wav", "./borsch.wav")))
target_voices = [load_for_knn_vc(f"./target_voices/{p}") for p in os.listdir("./target_voices")]

display(HTML("Original"))
for orig_voice in orig_voices: display(Audio(orig_voice, rate=KNN_VC_SR))
    
display(HTML("Target Voice"))
display(Audio(target_voices[0], rate=KNN_VC_SR))

### TTS

In [ ]:
g2p = en.G2P(trf=False, fallback=espeak.EspeakFallback(british=False))

In [ ]:
kokoro = Kokoro("./models/kokoro-v1.0.fp16-gpu.onnx", "./models/voices-v1.0.bin")

In [ ]:
phonemes_paragraphs = [g2p(t)[0] for t in text.split("\n\n")]

for voice in [ 
    # english-native
    "af_sky", "am_liam", "am_puck", "am_santa",
    # accents
    "hm_omega",
]:
    paragraphs = []
    for phonemes in phonemes_paragraphs:
        paragraph, _ = kokoro.create(phonemes, is_phonemes=True, voice=voice, speed=1.0)
        paragraphs.append(paragraph)
    if paragraphs: display(Audio(np.concat([np.pad(p, int(0.3*KOKORO_SR)) for p in paragraphs]), rate=KOKORO_SR))


### VC

In [ ]:
knn_vc = hub.load(
    "bshall/knn-vc", "knn_vc",
    pretrained=True, prematched=True,
    trust_repo=True,
    device=torch.device(torch.cuda.is_available() and "cuda" or "cpu"),
)

In [ ]:
query_seqs = list(map(knn_vc.get_features, orig_voices))

In [ ]:
matching_seq = knn_vc.get_matching_set(target_voices)

In [ ]:
for query_seq in query_seqs: 
    converted = knn_vc.match(query_seq, matching_seq, topk=4)
    display(Audio(converted, rate=KNN_VC_SR))